In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(ROOT)
from src.Orchestration.helpers import (
    fetch_direct_child_runs
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.Orchestration.helpers import fetch_direct_child_runs

In [ ]:
df, exp_name = fetch_direct_child_runs("2", "7e99f7fdc8a44f1b9461c3c1bdc6ec46")
print(exp_name)
print(df.shape)

df[['metrics.RMSE', 'metrics.Negative_log_likelihood',
       'metrics.Mean_Winkler_Score', 'metrics.Winkler_Coverage']]

In [ ]:

architectures = ["[16, 16]", "[32, 32]", "[32, 32, 32]", "[16, 16, 16, 16]"]
baseline_arch = "[16, 16]"

run_specs = [
    {
        "dataset": "fiat500",
        "experiment_id": "5",
        "run_id": "425666d19b9c4027a930589b8a6c63f1",
    },
    {
        "dataset": "healthcare_insurance",
        "experiment_id": "3",
        "run_id": "4a553a4264b54f3fb6b43216d4d7d444",
    },
    {
        "dataset": "wine_quality",
        "experiment_id": "4",
        "run_id": "c963c22d0c7b4f13a6ee0dc353b3ba1c",
    },
    {
        "dataset": "miami_housing",
        "experiment_id": "2",
        "run_id": "7e99f7fdc8a44f1b9461c3c1bdc6ec46",
    },
]


def build_best_per_architecture_df(run_specs, architectures):
    rows = []

    for spec in run_specs:
        df, exp_name = fetch_direct_child_runs(
            experiment_id=spec["experiment_id"],
            run_id=spec["run_id"],
        )

        work = df[[
            "run_id",
            "params.hidden_layers",
            "metrics.RMSE",
            "metrics.Negative_log_likelihood",
            "metrics.Mean_Winkler_Score",
        ]].copy()

        work = work.rename(columns={
            "params.hidden_layers": "architecture",
            "metrics.RMSE": "rmse",
            "metrics.Negative_log_likelihood": "nll",
            "metrics.Mean_Winkler_Score": "winkler",
        })

        work["architecture"] = work["architecture"].astype(str)
        work = work[work["architecture"].isin(architectures)]

        best_idx = work.groupby("architecture")["winkler"].idxmin()
        best_rows = work.loc[best_idx].copy()

        best_rows["architecture"] = pd.Categorical(
            best_rows["architecture"],
            categories=architectures,
            ordered=True,
        )
        best_rows = best_rows.sort_values("architecture")
        best_rows["dataset"] = spec["dataset"]

        rows.append(best_rows)

    return pd.concat(rows, ignore_index=True)


best_runs_df = build_best_per_architecture_df(run_specs, architectures)


def build_metric_trajectory_from_best(best_runs_df, metric_col, baseline_arch, lower_is_better=True):
    result_rows = []

    for dataset_name, group in best_runs_df.groupby("dataset"):
        group = group.copy()

        baseline_value = group.loc[
            group["architecture"] == baseline_arch, metric_col
        ].iloc[0]

        if lower_is_better:
            group["improvement_pct"] = (
                (baseline_value - group[metric_col]) / abs(baseline_value)
            ) * 100
        else:
            group["improvement_pct"] = (
                (group[metric_col] - baseline_value) / abs(baseline_value)
            ) * 100

        row = (
            group.set_index("architecture")["improvement_pct"]
            .reindex(architectures)
            .rename(dataset_name)
        )
        result_rows.append(row)

    return pd.DataFrame(result_rows)


trajectory_rmse = build_metric_trajectory_from_best(
    best_runs_df, metric_col="rmse", baseline_arch=baseline_arch, lower_is_better=True
)

trajectory_nll = build_metric_trajectory_from_best(
    best_runs_df, metric_col="nll", baseline_arch=baseline_arch, lower_is_better=True
)

trajectory_winkler = build_metric_trajectory_from_best(
    best_runs_df, metric_col="winkler", baseline_arch=baseline_arch, lower_is_better=True
)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("paper", font_scale=1.2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=False)
axes = axes.flatten()

metric_plots = [
    ("RMSE", trajectory_rmse, axes[0]),
    ("Negative Log-Likelihood (NLL)", trajectory_nll, axes[1]),
    ("Winkler Score", trajectory_winkler, axes[2]),
]

dataset_labels = {
    "fiat500": "Fiat 500",
    "healthcare_insurance": "Health Insurance",
    "miami_housing": "Miami Housing",
    "wine_quality": "Wine Quality",
}

palette = sns.color_palette("colorblind", 4)
dataset_colors = {
    "fiat500": palette[0],
    "healthcare_insurance": palette[1],
    "miami_housing": palette[2],
    "wine_quality": palette[3],
}

dataset_markers = {
    "fiat500": "o",
    "healthcare_insurance": "s",
    "miami_housing": "^",
    "wine_quality": "D",
}

datasets = trajectory_rmse.index.tolist()

for plot_idx, (metric_name, trajectory_df, ax) in enumerate(metric_plots):
    for dataset_name in datasets:
        ax.plot(
            architectures,
            trajectory_df.loc[dataset_name, architectures].values,
            marker=dataset_markers[dataset_name],
            markersize=8,
            linewidth=2.5,
            label=dataset_labels[dataset_name],
            color=dataset_colors[dataset_name],
        )

    ax.set_title(metric_name, fontsize=14, fontweight="bold")
    ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)

    ax.set_title(metric_name, fontsize=14, fontweight="bold")

    if plot_idx in (0, 2):
        ax.set_ylabel("% Improvement", fontsize=13)
    else:
        ax.set_ylabel("")

    if plot_idx == 2:
        ax.set_xlabel("Network Architecture", fontsize=13)
    else:
        ax.set_xlabel("")

    ax.tick_params(axis="x", rotation=35, labelsize=11)
    ax.tick_params(axis="y", labelsize=11)
    ax.grid(True, alpha=0.5)
legend_ax = axes[3]
legend_ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Datasets",
    loc="lower right",
    ncol=2,
    frameon=True,
    fancybox=True,
    shadow=False,
    fontsize=11,
    title_fontsize=12,
)

fig.suptitle(
    "Best Run per Architecture Selected by Minimum Winkler Score\n(% Improvement vs [16, 16])",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("trajectory_best_per_architecture_2x2.pdf", format="pdf", bbox_inches="tight")
plt.show()